<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Momos20/si7011-catano/blob/main/evento3/sesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU en Colab** (T4 es suficiente) con acceso a Internet, ya que el modelo se descarga desde Hugging Face y TRL se instala con `pip`. Para descargar los datos desde Colab, configura el secreto `KAGGLE_API_TOKEN` antes de ejecutar el notebook.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

# Algunas imágenes de ejecución traen versiones antiguas de torchao incompatibles con transformers/peft.
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0)
torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

2026/10/28 01:01:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 01:01:37 INFO mlflow.store.db.utils: Updating database tables
2026/10/28 01:01:39 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.20.2 · trl 1.16.0 · peft 0.21.2 · MLflow 3.17.2 · cuda


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [3]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data')]

def kaggle_token():
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            token = userdata.get('KAGGLE_API_TOKEN')
            if token:
                os.environ['KAGGLE_API_TOKEN'] = token.strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None:
    assert kaggle_token(), 'Configura KAGGLE_API_TOKEN en los secretos de Colab.'
    import kagglehub
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])

assert ruta is not None, 'No encontré dolly_quijote_es.csv.'
df = pd.read_csv(ruta)

train_all = df[df.split == 'train'].reset_index(drop=True)
test = df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)

val = train_all.iloc[:40].reset_index(drop=True)
train = train_all.iloc[40:].reset_index(drop=True)

if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]

data_card = {
    'fuente': str(ruta),
    'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
    'train': len(train),
    'val': len(val),
    'test': len(test)
}
print(data_card)

GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo',
         'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1

print(f"respuestas de entrenamiento con algún giro de la época: "
      f"{train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")

e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

100%|██████████| 93.9k/93.9k [00:00<00:00, 356kB/s]

Extracting files...
{'fuente': '/root/.cache/kagglehub/datasets/juanmartinezv4399/si7011-s06-dolly-quijote/versions/2/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [4]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?
  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]
  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.
  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la 

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [5]:
def a_conversacion(fila, sistema=None):
    return {
        'prompt': mensajes(fila['instruction'], sistema),
        'completion': [
            {'role': 'assistant', 'content': fila['response_estilo']}
        ]
    }

In [6]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?', 'role': 'user'}], 'completion': [{'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.', 'role': 'assistant'}]}


## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

**Predicción:** espero que `base + estilo en el prompt` mejore frente al modelo base, aunque su $J$ quede más cerca de la base que de LoRA, porque el prompt orienta el estilo sin modificar los pesos del modelo.

In [10]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

args = SFTConfig(
    output_dir='sft_estilo',
    run_name='1-lora-r16',
    seed=0,
    completion_only_loss=True,
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    max_grad_norm=1.0,
    weight_decay=0.0,
    fp16=dev == 'cuda',
    bf16=False,
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='mlflow'
)

In [11]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el último equipo que se ha sumado a la Major League Soccer, y se llama St. Louis FC. Hizo su entrada en la competición en 2023, y en su primera temporada cosechó nada menos que cinco victorias seguidas, con una diferencia de goles asaz notable, de más de quince.<|im_end|>\n'


## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [22]:
t0 = time.time()
trainer.train()
tiempo_1 = time.time() - t0
print(f'{tiempo_1:.0f} s')

run_id = mlflow.last_active_run().info.run_id
CONTRATO = {
    'modelo': MODELO,
    'max_tokens_pregunta': 128,
    'max_nuevos': 150,
    'plantilla': 'chat de Qwen, sin sistema'
}

trainer.model.save_pretrained('lora_estilo')
json.dump(
    CONTRATO,
    open('lora_estilo/contrato.json', 'w'),
    ensure_ascii=False,
    indent=1
)

tam_adaptador_mb = sum(
    p.stat().st_size for p in Path('lora_estilo').rglob('*') if p.is_file()
) / 1e6

q_ref = test.instruction.iloc[0]

texto_base = tok.apply_chat_template(
    mensajes(q_ref),
    add_generation_prompt=True,
    tokenize=False
)

texto_estilo = tok.apply_chat_template(
    mensajes(q_ref, ESTILO),
    add_generation_prompt=True,
    tokenize=False
)

tokens_base = len(tok.encode(texto_base, add_special_tokens=False))
tokens_prompt = len(tok.encode(texto_estilo, add_special_tokens=False))
tokens_extra_prompt = tokens_prompt - tokens_base

with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_metric('tiempo_entrenamiento_s', tiempo_1)
    mlflow.log_metric('tam_adaptador_mb', tam_adaptador_mb)
    mlflow.log_metric('tokens_extra_prompt', tokens_extra_prompt)
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')

print('adaptador y contrato en MLflow, corrida', run_id)
print(f'adaptador: {tam_adaptador_mb:.1f} MB · prompt de estilo: +{tokens_extra_prompt} tokens por llamada')

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.692378,2.092935,1.974014,0.566625,102527.000000
20,1.778366,2.114386,2.005405,0.562730,115311.000000
30,1.832364,2.121590,1.980203,0.562925,127201.000000
40,1.538088,2.161905,1.908622,0.556698,139178.000000
50,1.551590,2.205094,1.798617,0.553929,152373.000000
60,1.495510,2.192090,1.840495,0.550247,164578.000000
70,1.495494,2.185363,1.849429,0.554175,176875.000000
72,1.495494,2.185310,1.849277,0.553944,179268.000000


42 s
adaptador y contrato en MLflow, corrida eef8ca09f3874e45a56626925196291f
adaptador: 8.7 MB · prompt de estilo: +38 tokens por llamada


In [13]:
def nll(datos, modelo_eval=None):
    """J de la respuesta sobre un Dataset; por defecto evalúa el LoRA principal."""
    m = trainer.model if modelo_eval is None else modelo_eval
    ev = SFTTrainer(
        model=m,
        processing_class=tok,
        train_dataset=datos,
        eval_dataset=datos,
        args=SFTConfig(
            output_dir='eval_tmp',
            per_device_eval_batch_size=8,
            max_length=384,
            fp16=dev == 'cuda',
            bf16=False,
            report_to='none'
        )
    )
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [14]:
def comparar(filas, modelo_eval=None):
    m = trainer.model if modelo_eval is None else modelo_eval

    with m.disable_adapter():
        j_base = nll(conjunto(filas), m)
        j_prompt = nll(conjunto(filas, ESTILO), m)

    j_lora = nll(conjunto(filas), m)

    return {
        'base': j_base,
        'base + estilo en el prompt': j_prompt,
        'LoRA': j_lora
    }

In [15]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'

with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, \
        '"base" se mide con el adaptador apagado y sin sistema.'

assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, \
    '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], \
    'La condición con el estilo en el prompt usa sistema=ESTILO.'

resultados = comparar(test)
tabla = pd.DataFrame([
    {'condición': k, 'J_test': v, 'perplejidad': math.exp(v)}
    for k, v in resultados.items()
])

distancia = resultados['base'] - resultados['LoRA']
cobertura_prompt = (
    100 * (resultados['base'] - resultados['base + estilo en el prompt']) / distancia
    if abs(distancia) > 1e-12 else float('nan')
)

with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
    mlflow.log_metric('cobertura_prompt_pct', cobertura_prompt)

print(tabla.round(3).to_string(index=False))
print(f'El prompt cubre {cobertura_prompt:.1f}% de la distancia entre base y LoRA.')

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (63 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.941690,0,1.847720,0.599086,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.942        6.971
El prompt cubre 13.3% de la distancia entre base y LoRA.


$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [16]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                              0.00                   76.20
base + estilo en el prompt   2.506       12.261                              0.00                   55.90
                      LoRA   1.942        6.971                              0.85                   57.55 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_ Reduciré únicamente el rango de LoRA de **r = 16 a r = 4**; espero una $J$ de test ligeramente mayor y un juez de estilo parecido, porque el estilo puede requerir poca capacidad adicional, aunque el adaptador pequeño tendrá menos parámetros entrenables.

In [18]:
modelo_2 = AutoModelForCausalLM.from_pretrained(
    MODELO,
    dtype=torch.float32
).to(dev)

lora_2 = LoraConfig(
    r=4,
    lora_alpha=32,
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

args_2 = SFTConfig(
    output_dir='sft_estilo_r4',
    run_name='2-lora-r4',
    seed=0,
    completion_only_loss=True,
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    max_grad_norm=1.0,
    weight_decay=0.0,
    fp16=dev == 'cuda',
    bf16=False,
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='mlflow'
)

trainer_2 = SFTTrainer(
    model=modelo_2,
    args=args_2,
    train_dataset=D_train,
    eval_dataset=D_val,
    processing_class=tok,
    peft_config=lora_2
)

print('Corrida 2: el único cambio experimental es r=16 → r=4')
trainer_2.model.print_trainable_parameters()

t0_2 = time.time()
trainer_2.train()
tiempo_2 = time.time() - t0_2
print(f'{tiempo_2:.0f} s')

run_id_2 = mlflow.last_active_run().info.run_id
resultados_2 = comparar(test, trainer_2.model)

tabla_2 = pd.DataFrame([
    {'condición': k, 'J_test': v, 'perplejidad': math.exp(v)}
    for k, v in resultados_2.items()
])

with mlflow.start_run(run_id=run_id_2):
    mlflow.log_param('cambio', 'r=4')
    mlflow.log_param('referencia', 'r=16')
    mlflow.log_metric('tiempo_entrenamiento_s', tiempo_2)
    mlflow.log_table(tabla_2, 'comparacion_test_r4.json')

print(tabla_2.round(3).to_string(index=False))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

... (30 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.959050,0,1.868179,0.595333,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.959        7.093


## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [19]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    carpeta = str(carpeta)
    contrato_path = Path(carpeta) / 'contrato.json'

    if not contrato_path.exists():
        raise ValueError('No encontré contrato.json en la carpeta del adaptador.')

    if carpeta not in _CACHE:
        contrato = json.load(open(contrato_path, encoding='utf-8'))
        tok_local = AutoTokenizer.from_pretrained(contrato['modelo'])
        base = AutoModelForCausalLM.from_pretrained(
            contrato['modelo'],
            dtype=torch.float32
        ).to(dev)
        modelo_local = PeftModel.from_pretrained(base, carpeta).to(dev)
        modelo_local.eval()
        _CACHE[carpeta] = (contrato, tok_local, modelo_local)

    contrato, tok_local, modelo_local = _CACHE[carpeta]

    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser un texto no vacío.')

    n_tokens = len(tok_local.encode(pregunta, add_special_tokens=False))
    if n_tokens > contrato['max_tokens_pregunta']:
        raise ValueError(
            f"La pregunta tiene {n_tokens} tokens; el máximo es "
            f"{contrato['max_tokens_pregunta']}."
        )

    x = tok_local.apply_chat_template(
        mensajes(pregunta),
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True
    ).to(dev)

    fin_local = tok_local.convert_tokens_to_ids('<|im_end|>')
    with torch.no_grad():
        out = modelo_local.generate(
            **x,
            max_new_tokens=contrato['max_nuevos'],
            do_sample=False,
            eos_token_id=fin_local,
            pad_token_id=tok_local.pad_token_id or tok_local.eos_token_id
        )

    return tok_local.decode(
        out[0, x['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

In [20]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

1. Consistencia ✓ · La bicicleta de gravilla, os digo yo, es una máquina que se utiliza para recorrer senderos en bicicletas, y que puede ser utilizada tanto por [...]
1. Consistencia ✓ · Aqueste es el chiste que os dirijo: Aqueste ave, que se llama "La Antártica", no puede hube de ser tan original como los aves de la Tierra, pues [...]
1. Consistencia ✓ · Aqueste es el nombre de la película que os digo que os recomiendo: "La muerte de la princesa Sofia", de Christopher Nolan, una obra magnífica [...]
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta tiene 402 tokens; el máximo es 128.


## 9. Preguntas

1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

**1.** Sí se cumplió la predicción, porque el prompt mejoró \(J\) de **2.593 a 2.506**, aunque quedó mucho más cerca de la base que de LoRA, cuya \(J\) fue **1.942**; en consecuencia, el prompt cubrió solo **13.3 %** de la distancia entre base y LoRA.

**2.** Coinciden en gran parte, ya que LoRA obtiene la menor \(J\), **1.942**, y además el juez detecta giros de época en **85 %** de sus respuestas; sin embargo, no siempre coinciden, pues el prompt mejora \(J\) hasta **2.506** y aun así el juez marca **0 %** de giros, como ocurre en el chiste del ave antártica, cuya respuesta no presenta claramente el estilo solicitado.

**3.** No se pueden comparar directamente porque \(J\) es una pérdida promedio por token y aquí se evalúan respuestas objetivo con estilo; además, el notebook 6.3 utiliza otra condición de entrenamiento, mientras que otro tokenizador dividiría los mismos textos en unidades diferentes y cambiaría la escala de la pérdida.

**4.** El prompt no requiere entrenamiento, aunque añade **38 tokens en cada llamada**; en cambio, LoRA necesitó aproximadamente **54 s de entrenamiento** y un adaptador de **8.7 MB**, por lo que el prompt conviene cuando el estilo se usa ocasionalmente o debe cambiar con frecuencia, mientras que LoRA resulta más conveniente cuando ese comportamiento se reutiliza muchas veces.

**5.** Actualmente `responder_raw` aceptaría una pregunta en inglés siempre que sea texto no vacío y no supere los **128 tokens**, porque el contrato no valida el idioma; por tanto, agregaría `idioma='es'` al contrato junto con una validación que rechace o redirija entradas fuera del dominio esperado.